<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Visualization**


Estimated time needed: **45** minutes


In this lab, you will focus on data visualization. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


After completing this lab, you will be able to:


-   Visualize the distribution of data.

-   Visualize the relationship between two features.

-   Visualize composition and comparison of data.




## Demo: How to work with database


Download the database file.


In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

**Install and Import Necessary Python Libraries**

Ensure that you have the required libraries installed to work with SQLite and Pandas:


In [ ]:
!pip install pandas 
!pip install matplotlib
!pip install seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

**Read the CSV File into a Pandas DataFrame**

Load the Stack Overflow survey data into a Pandas DataFrame:


In [ ]:
# Read the CSV file
df = pd.read_csv('survey-data.csv')

# Display the first few rows of the data
df.head()


**Create a SQLite Database and Insert the Data**

Now, let's create a new SQLite database (`survey-data.sqlite`) and insert the data from the DataFrame into a table using the sqlite3 library:


In [ ]:
import sqlite3

# Create a connection to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Write the dataframe to the SQLite database
df.to_sql('main', conn, if_exists='replace', index=False)


# Close the connection
conn.close()


**Verify the Data in the SQLite Database**
Verify that the data has been correctly inserted into the SQLite database by running a simple query:


In [ ]:
# Reconnect to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Run a simple query to check the data
QUERY = "SELECT * FROM main LIMIT 5"
df_check = pd.read_sql_query(QUERY, conn)

# Display the results
print(df_check)


## Demo: Running an SQL Query


Count the number of rows in the table named 'main'


In [ ]:
QUERY = """
SELECT COUNT(*) 
FROM main
"""
df = pd.read_sql_query(QUERY, conn)
df.head()


## Demo: Listing All Tables


To view the names of all tables in the database:


In [ ]:
QUERY = """
SELECT name as Table_Name FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


## Demo: Running a Group By Query
    
For example, you can group data by a specific column, like Age, to get the count of respondents in each age group:


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count
FROM main
GROUP BY Age
ORDER BY Age
"""
pd.read_sql_query(QUERY, conn)


## Demo: Describing a table

Use this query to get the schema of a specific table, main in this case:


In [ ]:
table_name = 'main'

QUERY = """
SELECT sql FROM sqlite_master 
WHERE name= '{}'
""".format(table_name)

df = pd.read_sql_query(QUERY, conn)
print(df.iat[0,0])


## Hands-on Lab


### Visualizing the Distribution of Data

**Histograms**

Plot a histogram of CompTotal (Total Compensation).


In [ ]:
query_comp = "SELECT CompTotal FROM main"
df_comp = pd.read_sql_query(query_comp, conn)


df_comp = df_comp.dropna()


plt.figure(figsize=(10, 6))
plt.hist(df_comp['CompTotal'], bins=30, color='skyblue', edgecolor='black')


plt.title('Distribution of Total Compensation')
plt.xlabel('Total Compensation')
plt.ylabel('Number of Respondents')
plt.grid(axis='y', alpha=0.75)


plt.show()

**Box Plots**

Plot a box plot of Age.


In [ ]:
query_age = "SELECT Age FROM main"
df_age = pd.read_sql_query(query_age, conn)

df_age = df_age.dropna()

plt.figure(figsize=(8, 6))
sns.boxplot(x=df_age['Age'], color='lightgreen')

plt.title('Box Plot of Respondent Age Distribution')
plt.xlabel('Age (Years)')

plt.show()

### Visualizing Relationships in Data

**Scatter Plots**

Create a scatter plot of Age and WorkExp.


In [ ]:
query_scatter = "SELECT Age, WorkExp FROM main"
df_scatter = pd.read_sql_query(query_scatter, conn)

df_scatter = df_scatter.dropna(subset=['Age', 'WorkExp'])

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_scatter, x='Age', y='WorkExp', alpha=0.6, color='coral')

plt.title('Relationship Between Age and Work Experience')
plt.xlabel('Age (Years)')
plt.ylabel('Work Experience (Years)')
plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

**Bubble Plots**

Create a bubble plot of `TimeSearching` and `Frustration` using the Age column as the bubble size.


In [ ]:
query_bubble = "SELECT TimeSearching, Frustration, Age FROM main"
df_bubble = pd.read_sql_query(query_bubble, conn)


df_bubble['Age'] = pd.to_numeric(df_bubble['Age'], errors='coerce')
df_bubble = df_bubble.dropna(subset=['TimeSearching', 'Frustration', 'Age'])


plt.figure(figsize=(10, 6))
plt.scatter(
    x=df_bubble['TimeSearching'],
    y=df_bubble['Frustration'],
    s=df_bubble['Age'] * 4,
    alpha=0.5,
    color='purple',
    edgecolors='w'
)

plt.title('Bubble Plot of Time Searching vs Frustration (Bubble Size = Age)')
plt.xlabel('Time Searching')
plt.ylabel('Frustration Level')
plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

### Visualizing Composition of Data

**Pie Charts**

Create a pie chart of the top 5 databases(`DatabaseWantToWorkWith`) that respondents wish to learn next year.


In [ ]:
QUERY = "SELECT DatabaseWantToWorkWith FROM main"
df_main = pd.read_sql_query(QUERY, conn)

all_databases = df_main['DatabaseWantToWorkWith'].dropna().str.split(';')
expanded_df = all_databases.explode()

top_5_databases = expanded_df.value_counts().head(5)


colors_list = ['#5cb85c', '#5bc0de', '#d9534f', '#f0ad4e', '#337ab7']


top_5_databases.plot(
    kind='pie',
    figsize=(10, 6),
    autopct='%1.1f%%',
    startangle=90,
    shadow=True,
    colors=colors_list
)


plt.title('Top 5 Databases Respondents Wish to Learn Next Year', fontsize=14, fontweight='bold', pad=20)
plt.ylabel('')
plt.axis('equal')
plt.tight_layout()

plt.show()

**Stacked Charts** 

Create a stacked bar chart of median `TimeSearching` and `TimeAnswering` for the age group 30 to 35.


In [ ]:
query_stacked = """
SELECT TimeSearching, TimeAnswering, Age 
FROM main 
WHERE Age BETWEEN 30 AND 35
"""
df_stacked = pd.read_sql_query(query_stacked, conn)

df_med = df_stacked.groupby('Age').median()

df_med.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#4ea8de', '#56cfe1'])

plt.title('Median Time Searching and Time Answering for Ages 30-35', fontsize=14, weight='bold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Median Hours / Units of Time', fontsize=12)
plt.xticks(rotation=0)  
plt.legend(['Time Searching', 'Time Answering'])
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

### Visualizing Comparison of Data

**Line Chart**

Plot the median `CompTotal` for all ages from 45 to 60.


In [ ]:
query_line = """
SELECT CompTotal, Age 
FROM main 
WHERE Age BETWEEN 45 AND 60
"""
df_line = pd.read_sql_query(query_line, conn)

df_line_med = df_line.groupby('Age').median()

plt.figure(figsize=(10, 6))
plt.plot(df_line_med.index, df_line_med['CompTotal'], marker='o', color='teal', linewidth=2)

plt.title('Median Total Compensation for Ages 45-60', fontsize=14, weight='bold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Median Compensation', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)


plt.show()

**Bar Chart**

Create a horizontal bar chart using the `MainBranch` column.


In [ ]:
query_barh = """
SELECT MainBranch, COUNT(*) as Frequency 
FROM main 
GROUP BY MainBranch
ORDER BY Frequency ASC
"""
df_barh = pd.read_sql_query(query_barh, conn)

plt.figure(figsize=(10, 5))
plt.barh(df_barh['MainBranch'], df_barh['Frequency'], color=['#3a86c8', '#4ea8de'], edgecolor='black', height=0.5)

plt.title('Frequency of Main Branches Among Respondents', fontsize=14, weight='bold')
plt.xlabel('Number of Respondents', fontsize=12)
plt.ylabel('Main Branch', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()

plt.show()


### Summary


In this lab, you focused on extracting and visualizing data from an RDBMS using SQL queries and SQLite. You applied various visualization techniques, including:

- Histograms to display the distribution of CompTotal.
- Box plots to show the spread of ages.
- Scatter plots and bubble plots to explore relationships between variables like Age, WorkExp, `TimeSearching` and `TimeAnswering`.
- Pie charts and stacked charts to visualize the composition of data.
- Line charts and bar charts to compare data across categories.


### Close the Database Connection

Once the lab is complete, ensure to close the database connection:


In [ ]:
conn.close()

## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
